# Lista 5 — Zadanie 5: Eksploracja parametrów LLM (30 pkt)

Porównujemy co najmniej **dwa aspekty**:
1. **Temperatura** generacji (0.0 vs 0.1 vs 0.7)
2. **Prompt** — prosty vs szczegółowy (definicje klas po polsku)
3. **Parsowanie** — regex vs `JsonOutputParser` (Pydantic)

Wyniki zestawiamy w tabeli porównawczej.

In [ ]:
import sys

!{sys.executable} -m pip install -q torch transformers datasets scikit-learn pandas langchain-core langchain-huggingface pydantic accelerate

In [ ]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import json
import re

import torch
import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions

## Krok 1: Konfiguracja

Ze względu na czas inferencji LLM, eksperymenty uruchamiamy na podzbiorze. Ustaw `SAMPLE_SIZE = None` dla pełnego testu.

In [ ]:
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SAMPLE_SIZE = 80  # None = cały zbiór

examples = load_polemo_test()
if SAMPLE_SIZE is not None:
    examples = examples[:SAMPLE_SIZE]

sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]
print(f"Próbek: {len(sentences)} | GPU: {torch.cuda.is_available()}")

## Krok 2: Definicje promptów

In [ ]:
PROMPT_SIMPLE = """Classify the text sentiment into one of three classes: positive, negative, neutral.
Reply with only one word.

Text: {text}
Class:"""

PROMPT_DETAILED = """Jesteś klasyfikatorem wydźwięku polskich recenzji.
Przypisz tekst do dokładnie jednej klasy:
- positive — opinia jednoznacznie pozytywna
- negative — opinia jednoznacznie negatywna
- neutral — opinia opisowa, bez wyraźnych emocji

Odpowiedz jednym słowem (positive, negative lub neutral).

Tekst: {text}
Klasa:"""

PROMPT_JSON = """Jesteś klasyfikatorem wydźwięku. Zwróć odpowiedź jako JSON.

Tekst: {text}

{format_instructions}"""


class SentimentResult(BaseModel):
    sentiment: str = Field(description="One of: positive, negative, neutral")


json_parser = JsonOutputParser(pydantic_object=SentimentResult)
json_format_instructions = json_parser.get_format_instructions()

## Krok 3: Funkcje do eksperymentów

In [ ]:
_loaded_model = None
_loaded_tokenizer = None


def get_llm(temperature=0.1, max_new_tokens=15):
    """Ładuje model raz i tworzy pipeline z podaną temperaturą."""
    global _loaded_model, _loaded_tokenizer

    if _loaded_model is None:
        _loaded_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
        _loaded_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )

    hf_pipe = pipeline(
        "text-generation",
        model=_loaded_model,
        tokenizer=_loaded_tokenizer,
        temperature=temperature,
        do_sample=temperature > 0,
        max_new_tokens=max_new_tokens,
        pad_token_id=_loaded_tokenizer.eos_token_id,
    )
    return HuggingFacePipeline(pipeline=hf_pipe)


def parse_text_answer(answer: str) -> str:
    """Parsuje zwykłą odpowiedź tekstową LLM."""
    mapped = map_text_to_class(answer)
    return mapped if mapped else "neutral"


def parse_json_answer(answer: str) -> str:
    """Parsuje odpowiedź JSON za pomocą JsonOutputParser."""
    try:
        parsed = json_parser.parse(answer)
        sentiment = parsed.get("sentiment", "")
        return parse_text_answer(sentiment)
    except Exception:
        # Fallback: spróbuj wyciągnąć JSON regexem
        match = re.search(r'\{[^}]+\}', answer)
        if match:
            try:
                data = json.loads(match.group())
                return parse_text_answer(data.get("sentiment", ""))
            except json.JSONDecodeError:
                pass
        return parse_text_answer(answer)


def run_llm_experiment(name, prompt_template, temperature, parse_fn, use_json_format=False):
    """Uruchamia jeden eksperyment LLM i zwraca metryki."""
    llm = get_llm(temperature=temperature)

    if use_json_format:
        prompt = PromptTemplate.from_template(prompt_template)
        chain = prompt | llm
        invoke_kwargs = {"text": None, "format_instructions": json_format_instructions}
    else:
        prompt = PromptTemplate.from_template(prompt_template)
        chain = prompt | llm
        invoke_kwargs = None

    y_pred = []
    for sentence in tqdm(sentences, desc=name):
        if use_json_format:
            answer = chain.invoke({"text": sentence, "format_instructions": json_format_instructions})
            y_pred.append(parse_fn(answer))
        else:
            answer = chain.invoke({"text": sentence})
            y_pred.append(parse_fn(answer))

    metrics = evaluate_predictions(y_true, y_pred)
    return {
        "eksperyment": name,
        "temperature": temperature,
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_weighted": metrics["f1_weighted"],
    }

## Eksperyment A: Wpływ temperatury (prosty prompt)

In [ ]:
TEMPERATURES = [0.0, 0.1, 0.7]
temp_results = []

for temp in TEMPERATURES:
    result = run_llm_experiment(
        name=f"temp={temp}",
        prompt_template=PROMPT_SIMPLE,
        temperature=temp,
        parse_fn=parse_text_answer,
    )
    temp_results.append(result)
    print(f"temp={temp}: accuracy={result['accuracy']:.4f}, f1_macro={result['f1_macro']:.4f}")

## Eksperyment B: Wpływ promptu (temperatura=0.1)

In [ ]:
prompt_results = []

for prompt_name, prompt_text in [("prosty", PROMPT_SIMPLE), ("szczegółowy", PROMPT_DETAILED)]:
    result = run_llm_experiment(
        name=f"prompt={prompt_name}",
        prompt_template=prompt_text,
        temperature=0.1,
        parse_fn=parse_text_answer,
    )
    result["prompt"] = prompt_name
    prompt_results.append(result)
    print(f"prompt={prompt_name}: accuracy={result['accuracy']:.4f}, f1_macro={result['f1_macro']:.4f}")

## Eksperyment C: Parsowanie JSON (JsonOutputParser)

In [ ]:
json_result = run_llm_experiment(
    name="parsowanie=JSON",
    prompt_template=PROMPT_JSON,
    temperature=0.1,
    parse_fn=parse_json_answer,
    use_json_format=True,
)
json_result["prompt"] = "JSON"
print(f"JSON parser: accuracy={json_result['accuracy']:.4f}, f1_macro={json_result['f1_macro']:.4f}")

## Krok 4: Tabela porównawcza wszystkich eksperymentów

In [ ]:
all_results = temp_results + prompt_results + [json_result]
comparison_df = pd.DataFrame(all_results)
comparison_df.sort_values("f1_macro", ascending=False)

## Podsumowanie (do raportu)

| Aspekt | Wnioski |
|--------|--------|
| **Temperatura** | Niska (0.0–0.1) daje stabilniejsze, powtarzalne odpowiedzi. Wysoka (0.7) zwiększa losowość. |
| **Prompt** | Szczegółowy prompt z definicjami klas po polsku może poprawić trafność na polskich recenzjach. |
| **JsonOutputParser** | Ustrukturyzowane parsowanie redukuje błędy mapowania, ale wymaga dłuższej odpowiedzi modelu. |

**Porównanie encoder vs decoder:** encoder (zad. 2–3) jest szybszy i zwykle dokładniejszy na masowej klasyfikacji; LLM daje większą elastyczność (zmiana promptu bez retrenowania), ale kosztuje więcej czasu i zasobów.